In [ ]:
import numpy as np
import pandas as pd

from matplotlib import pyplot as plt
import json 
import scanpy as sc 
from PIL import Image
import cv2
from shapely import geometry
from shapely.geometry import Polygon
import rasterio.features
from scipy.sparse import issparse

### Find matched spots

In [ ]:
adata_8um = sc.read_h5ad('../DBiT_imputation-CellViT-VisiumHD_HumanKidney-8x-spatialDE.h5ad')
adata_8um_original = sc.read_h5ad('../../Visium/HumanKidney_Visium_HD_8um.h5ad')

In [ ]:
selected_spots=[]
dist_x = []
dist_y =[]
for i in range(adata_8um.obsm['spatial'].shape[0]):
    tmp_dist = (adata_8um_original.obsm['spatial'][:, 0] - adata_8um.obs['x'][i])**2 + (adata_8um_original.obsm['spatial'][:, 1] - adata_8um.obs['y'][i])**2
    tmp_index = np.where(tmp_dist == np.min(tmp_dist))[0][0]
    selected_spots.append(tmp_index)
    dist_x.append(np.abs(adata_8um_original.obsm['spatial'][tmp_index, 0] - adata_8um.obs['x'][i]))
    dist_y.append(np.abs(adata_8um_original.obsm['spatial'][tmp_index, 1] - adata_8um.obs['y'][i]))
    
cut_off = 1
matched_spots = (np.array(dist_x)<cut_off)*(np.array(dist_y)<cut_off)
final_selected_spots = np.array(selected_spots)[matched_spots]

In [ ]:
adata_original = adata_8um_original[final_selected_spots, adata_8um.var_names]
adata_predict = adata_8um[matched_spots, :].copy()

In [ ]:
orig_exp = adata_original.X.copy()
adata_original.layers['counts'] = orig_exp

In [ ]:
import os
import argparse
import sys

# Get the absolute path to the directory containing the script to be imported
script_dir = os.path.abspath("/gpfs/ycga/work/fan/yl2499/istar")
# Add the directory to sys.path
sys.path.append(script_dir)
from iStar.utils import load_image
from iStar.plot_imputed import plot_super

In [ ]:
def inside_polygon(xy, polygon):
    points_inside_polygon = list(map(polygon.contains, map(geometry.Point, xy)))

    return np.array(points_inside_polygon)

def get_region(ad, polygon_coors):
    if type(polygon_coors) in (geometry.polygon.Polygon, geometry.multipolygon.MultiPolygon):
        polygon = polygon_coors
    else:
        assert np.array_equal(
            polygon_coors[0], polygon_coors[-1]
        ), "Polygon is not closed. Closing it by connecting the first and last vertices."
        #polygon = [(1, 5), (2, 5), (2, 2), (1, 2), (1, 5)]
        polygon = geometry.Polygon(polygon_coors)

    xy = ad.obsm['spatial']
    ad_roi = ad[inside_polygon(xy, polygon)]
    return ad_roi.copy()

In [ ]:
def get_adata_layer(adata, layer_key=None):
    cnts = adata.X if layer_key is None else adata.layers[layer_key]
    return cnts


def get_adata_layer_array(adata, layer_key=None):
    cnts = get_adata_layer(adata, layer_key=layer_key)

    if issparse(cnts):
        cnts = cnts.toarray()

    return cnts


def rescale(value, scalefactor):
    return int(np.round(value*scalefactor))

def resize_array(arr, target_shape, interpolation=cv2.INTER_LINEAR):
    import cv2
    return cv2.resize(arr, target_shape, interpolation=interpolation) 

def get_downscaled_im(gene_array, scalefactor, locs, image_size):
    xmin, ymin = 1, 1
    xmax, ymax = image_size

    grid_sizes = (rescale(ymax, scalefactor), rescale(xmax, scalefactor))
    grid_values = np.zeros(grid_sizes)

    # Populate the grid with values from the spatial data
    x = locs[:, 0]
    y = locs[:, 1]

    col_index = np.digitize(x, np.linspace(xmin, xmax, grid_sizes[1])).astype(int)
    row_index = np.digitize(y, np.linspace(ymin, ymax, grid_sizes[0])).astype(int)

    # Assign the value to the corresponding grid cell
    grid_values[row_index - 1, col_index - 1] += gene_array[:, 0]

    grid_values[grid_values < 0] = 0
    return grid_values

def get_downscaled_roi_im(gene, ad, image_size, scalefactor, key_word = 'spatial', roi=None, layer_key=None):
    gene_array = get_adata_layer_array(ad[:, gene], layer_key)
    #scalefactor = get_scalefactors(ad)['tissue_hires_scalef']
    locs = ad.obsm[key_word]
    
    grid_values = get_downscaled_im(gene_array, scalefactor, locs, image_size)

    if roi is None:
        return grid_values
    
    roi_downscale = (rescale(roi[2], scalefactor), rescale(roi[3], scalefactor), rescale(roi[0], scalefactor), rescale(roi[1],scalefactor))
    gene_mat_roi = grid_values[roi_downscale[0]: roi_downscale[1], roi_downscale[2]:roi_downscale[3]]
    
    return gene_mat_roi


### VisiumHD

In [ ]:
adata_predict.obsm['spatial_he'] = adata_predict.obsm['spatial'].copy()
adata_original.obsm['spatial_he'] = adata_original.obsm['spatial'].copy()

In [ ]:
visium_coords = np.array([np.array(adata_original.obs['array_col']), np.array(adata_original.obs['array_row'])]).T
adata_original.obsm['spatial'] = visium_coords
adata_predict.obsm['spatial'] = visium_coords

In [ ]:
predicted_gene = np.array(adata_predict.layers['counts'])
true_gene = np.array(adata_original.layers['counts'])
predicted_gene_x = np.array(adata_predict.obsm['spatial'][:, 0])
predicted_gene_y = np.array(adata_predict.obsm['spatial'][:, 1])
true_gene_x = adata_original.obsm['spatial'][:, 0]
true_gene_y = adata_original.obsm['spatial'][:, 1]

In [ ]:
# keep regions where we have prediction from visium
rect = cv2.minAreaRect(adata_predict.obsm['spatial'].astype(int))
box = cv2.boxPoints(rect)
box = np.int0(box)
bbox = Polygon(box)
xenium_data_sub_intersect = get_region(adata_original, bbox)

# keep regions where we also have xenium coverage
rect2 = cv2.minAreaRect(adata_original.obsm['spatial'].astype(int))
box2 = cv2.boxPoints(rect2)
box2 = np.int0(box2)
 
image_size = (np.min([np.max(predicted_gene_x), np.max(true_gene_x)]).astype(int), 
              np.min([np.max(predicted_gene_y), np.max(true_gene_y)]).astype(int))
ad = adata_original
scalefactor = 1.#get_scalefactors(ad)['tissue_hires_scalef']
bbox2 = Polygon(box2 * scalefactor)

xmin, ymin = 1, 1
xmax, ymax = image_size
grid_sizes = (rescale(ymax, scalefactor), rescale(xmax, scalefactor))
mask = rasterio.features.rasterize([bbox2], out_shape=grid_sizes)

In [ ]:
Image.fromarray(mask.astype(bool)).save("visiumhd-common-mask_hires.png")

In [ ]:
import os
import sys
# Get the absolute path to the directory containing the script to be imported
script_dir = os.path.abspath("/gpfs/ycga/work/fan/yl2499/istar")
# Add the directory to sys.path
sys.path.append(script_dir)
from iStar.utils import load_image
from iStar.plot_imputed import plot_super

In [ ]:
mask = load_image(f'visiumhd-common-mask_hires.png') > 0

output_dir = "VisiumHD"
xen_output_dir = os.path.join(output_dir, f"Truth_spatialDE")
os.makedirs(xen_output_dir, exist_ok=True)

for gene in adata_original.var_names:
    xen_gene_roi = get_downscaled_roi_im(gene, adata_original, image_size, 1.0, layer_key='counts')
    xen_gene_roi[~mask] = np.nan
    plot_super(xen_gene_roi, os.path.join(xen_output_dir, f'{gene}.png'))

In [ ]:
mask = load_image(f'visiumhd-common-mask_hires.png') > 0

output_dir = "VisiumHD"
our_output_dir = os.path.join(output_dir, f"SuperFocus_results_spatialDE")
os.makedirs(our_output_dir, exist_ok=True)

for gene in adata_original.var_names:
    xen_gene_roi = get_downscaled_roi_im(gene, adata_predict, image_size, 1.0, 'spatial', layer_key='counts')
    xen_gene_roi[~mask] = np.nan
    plot_super(xen_gene_roi, os.path.join(our_output_dir, f'{gene}.png'))

Need tensorflow to run the code below

### MS-SSIM

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt 
import matplotlib as mpl
import pandas as pd
import pickle
from collections import Counter
import scanpy as sc
from PIL import Image
import os

In [ ]:
def gene_msssim(our_dir, xen_dir, gene, size):
    xen_image_path = os.path.join(xen_dir, f'{gene}.png')
    our_image_path = os.path.join(our_dir, f'{gene}.png')
    
    xen_image = Image.open(xen_image_path).convert('L')
    our_image = Image.open(our_image_path).convert('L')
    
    img1_np = np.array(xen_image)
    img2_np = np.array(our_image)

    img1_tf = tf.convert_to_tensor(img1_np, dtype=tf.float32)
    img2_tf = tf.convert_to_tensor(img2_np, dtype=tf.float32)

    # Add batch dimension if not already present (e.g., (height, width, channels) -> (1, height, width, channels))
    img1_tf = tf.expand_dims(img1_tf, axis=0)
    img2_tf = tf.expand_dims(img2_tf, axis=0)

    img1_tf = tf.expand_dims(img1_tf, axis=-1)
    img2_tf = tf.expand_dims(img2_tf, axis=-1)
    
    ms_ssim_value = tf.image.ssim_multiscale(img1_tf, img2_tf, max_val=255.0, filter_size=size)
    
    return ms_ssim_value.numpy()[0]

In [ ]:
methods = ['Our_results_spatialDE']

In [ ]:
ms_ssim_lists = []
output_dir = "VisiumHD"#"./ExpressionImages_breast_cancer-mask_hires"
xen_output_dir = os.path.join(output_dir, "Truth_spatialDE")#os.path.join(output_dir, "Xenium")
for train_group in train_groups:
    for method in methods:
        tmp_ms_ssim_list = []
        our_output_dir = os.path.join(output_dir, f"{method}")#os.path.join(output_dir, "Our_results")
        for gene in adata_original.var_names:
            ms_ssim = gene_msssim(our_output_dir, xen_output_dir, gene, 4)
            tmp_ms_ssim_list.append(ms_ssim)
        ms_ssim_lists.append(tmp_ms_ssim_list)

In [ ]:
import seaborn as sns
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
ms_ssim_array = np.array(ms_ssim_lists).reshape(-1, 1)
ms_ssim_pd = pd.DataFrame({'MS-SSIM': ms_ssim_array[:, 0], 'Method': np.tile(np.repeat(['SuperFocus'], 2010), 1), 
                           'Gene':np.tile(adata_original.var_names, 1)})
ms_ssim_pd

In [ ]:
np.nanmean(ms_ssim_pd['MS-SSIM'])

In [ ]:
np.nanstd(ms_ssim_pd['MS-SSIM'])

In [ ]:
df = pd.DataFrame(index=adata_original.var_names, columns=['SuperFocus'])
df['SuperFocus'] = np.array(ms_ssim_pd.loc[:, 'MS-SSIM'])

In [ ]:
import seaborn as sns
sns.set(font="Arial")
sns.set_theme(style='ticks')
sns.set_context("notebook", font_scale=2)

columns = ['SuperFocus']
fig = plt.figure(figsize=(5,5), dpi=300)
ax = sns.violinplot(df[columns], inner=None, alpha=0.8, cut=0, gap=0.3)
sns.pointplot(data=df[columns], #x="day", y="total_bill", 
              estimator="mean", errorbar="sd", 
              linestyle="none", marker="_", color='black', capsize=0.1, err_kws={'linewidth': 1}, markersize=10)
sns.stripplot(df[columns], size=0.8, jitter=0.1, dodge=False, legend="", alpha=1, ax=ax)
plt.ylabel('MS-SSIM')
#ax.set_ylim(0.5, 1)
ax.spines.right.set_color('none')
ax.spines.top.set_color('none')
for patch in ax.patches:
    r, g, b, a = patch.get_facecolor()
    patch.set_facecolor((r, g, b, .8))

### NRMSE

In [ ]:
output_dir = "VisiumHD"
xen_output_dir = os.path.join(output_dir, "Truth_spatialDE")
our_output_dir = os.path.join(output_dir, "SuperFocus_results_spatialDE")

from sklearn.metrics import root_mean_squared_error

def get_rmse_norm(gene):
    xen_image_path = os.path.join(xen_output_dir, f'{gene}.png')
    our_image_path = os.path.join(our_output_dir, f'{gene}.png')
    
    xen_image = np.array(Image.open(xen_image_path).convert('L'))
    our_image = np.array(Image.open(our_image_path).convert('L'))

    mean_xen = np.mean(xen_image)
    rmse_our = root_mean_squared_error(our_image, xen_image) /mean_xen
    
    return rmse_our

In [ ]:
df = pd.DataFrame(index=adata_original.var_names, columns=['SuperFocus'])
for gene in adata_original.var_names:
    df.loc[gene,  ['SuperFocus']] = get_rmse_norm(gene)

In [ ]:
import seaborn as sns
sns.set(font="Arial")
sns.set_theme(style='ticks')
sns.set_context("notebook", font_scale=2)

columns = ['SuperFocus']
fig = plt.figure(figsize=(5,5), dpi=300)
ax = sns.violinplot(df[columns], inner=None, alpha=0.8, cut=0)
sns.pointplot(data=df[columns], #x="day", y="total_bill", 
              estimator="mean", errorbar="sd", 
              linestyle="none", marker="_", color='black', capsize=0.1, err_kws={'linewidth': 1}, markersize=6)
sns.stripplot(df[columns], size=0.8, jitter=0.1, dodge=False, legend="", alpha=1, ax=ax)
plt.ylabel('NRMSE')
#ax.set_ylim(0.5, 1)
ax.spines.right.set_color('none')
ax.spines.top.set_color('none')
for patch in ax.patches:
    r, g, b, a = patch.get_facecolor()
    patch.set_facecolor((r, g, b, .8))

### ESL and RSSE

In [ ]:
with open('./VisiumHD_8um_counts_df.pickle', 'rb') as file:
    counts_est_8um, counts_orig_8um = pickle.load(file)
    
with open('./VisiumHD_64um_counts_df.pickle', 'rb') as file:
    counts_est_64um, counts_orig_64um = pickle.load(file)

In [ ]:
counts_est_8um

In [ ]:
import sys
import os

# Get the absolute path to the directory containing the script to be imported
script_dir = os.path.abspath("/gpfs/gibbs/project/fan/yl2499/code/SuperFocus")

# Add the directory to sys.path
sys.path.append(script_dir)

In [ ]:
from importlib import reload
import src.post_process.self_diagnosis_metrics
reload(src.post_process.self_diagnosis_metrics)
from src.post_process.self_diagnosis_metrics import genes_RSSE
from src.post_process.self_diagnosis_metrics import genes_ELS

In [ ]:
var_genes = counts_est_8um.var()

In [ ]:
rsse_64um = genes_RSSE(counts_est_64um, counts_orig_64um)
els_64um = genes_ELS(counts_est_64um, counts_orig_64um)

In [ ]:
rsse_8um = genes_RSSE(counts_est_8um, counts_orig_8um)
els_8um = genes_ELS(counts_est_8um, counts_orig_8um)

In [ ]:
from scipy.stats import spearmanr
spearmanr(rsse_8um, rsse_64um)

In [ ]:
from scipy.stats import spearmanr
spearmanr(els_8um, els_64um)

In [ ]:
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.left'] = True
plt.rcParams['axes.spines.bottom'] = True
fig = plt.figure(figsize=(6, 6), dpi=300)
plt.scatter(np.array(els_64um)[var_genes>0], np.array(els_8um)[var_genes>0], s=15.0, c='navy')
#plt.ylim(0, 4)
plt.text(0.18, 0.08, "r=0.85 \np-value<0.001", fontsize=16)
plt.xlabel("ESL (in-sample)", fontsize=16)
plt.ylabel("ESL (ground-truth at 8 um)", fontsize=16)
fig.savefig("VisiumHD_ESL.pdf")

In [ ]:
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.left'] = True
plt.rcParams['axes.spines.bottom'] = True
fig = plt.figure(figsize=(6, 6), dpi=300)
plt.scatter(np.array(rsse_64um)[var_genes>0]/np.max(np.array(rsse_64um)[var_genes>0]), np.array(rsse_8um)[var_genes>0]/np.max(np.array(rsse_8um)[var_genes>0]), s=15.0, c='navy')
plt.ylim(0, 0.4)
plt.text(0.9, 0.35, "r=0.92 \np-value<0.001", fontsize=16)
plt.xlabel("RSSE (in-sample)", fontsize=16)
plt.ylabel("RSSE (ground-truth at 8 um)", fontsize=16)
fig.savefig("VisiumHD_RSSE.pdf")